# SadTalker — Talking Head (صورة + صوت = إنسان واقعي يتكلم)

صورة بورتريه واحدة + ملف صوتي عربي = فيديو لإنسان واقعي يتكلم، بحركة رأس وملامح وغمزات.
مفتوح المصدر (Apache 2.0) — [SadTalker](https://github.com/OpenTalker/SadTalker) — ويشتغل على **GPU المجاني تاع Google Colab (T4)**.

**كيف تشغّل:** Runtime → Change runtime type → Hardware accelerator = **T4 GPU** → Save، ثم Runtime → Run all. أول تشغيل ياخذ حوالي 10 دقائق (تثبيت + تحميل الموديلات ~2GB). الفيديو الناتج يظهر في آخر خلية وتقدر تحمّله من مجلد `results/`.

الملفات التجريبية (بورتريه CC0 + تعليق عربي بصوت Piper/kareem) تُحمَّل تلقائياً من مستودع video-lab؛ أو ارفع ملفاتك الخاصة في خلية الرفع.

In [ ]:
# 1) Check the GPU (must be a T4 or better)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
import torch
print('CUDA available:', torch.cuda.is_available())

## 2) Setup (~5 min)
ينسخ SadTalker ويثبّت متطلباته. إذا فشلت هذه الخلية بخطأ NumPy (`VisibleDeprecationWarning`) راجع خلية «Troubleshooting» في الأسفل.

In [ ]:
print('Clone SadTalker and install requirements...')
!git clone -q https://github.com/OpenTalker/SadTalker
%cd SadTalker
# Colab Python 3.12/3.13 has no wheels for SadTalker's pinned numpy==1.23.4, scipy==1.10.1
# and scikit-image==0.19.3 — pip aborts the whole install there, so kornia etc. never
# install and generation fails with 'No module named kornia'. Strip those three pins.
# basicsr 1.4.2 also fails to BUILD in an isolated env on Python 3.13 (setup.py needs
# the already-installed torch/setuptools) — install it and the rest with
# --no-build-isolation so they build against Colab's environment.
!grep -v -E "^(numpy|scipy|scikit-image)==" requirements.txt > /tmp/req_colab.txt
!pip install -q --no-build-isolation basicsr==1.4.2
!pip install -q --no-build-isolation -r /tmp/req_colab.txt
# SadTalker uses np.VisibleDeprecationWarning, removed in NumPy 2 — pin NumPy 1.x
!pip install -q "numpy<2"
!python -c "import kornia, facexlib, gfpgan, basicsr, librosa; print('deps OK')"

## 3) Download the pre-trained models (~2 GB, ~1 min)

In [ ]:
print('Download pre-trained models...')
!rm -rf checkpoints
!bash scripts/download_models.sh

## 4) Get the demo portrait + Arabic voiceover
البورتريه CC0 (مجال عام) والتعليق مولّد محلياً بصوت Piper العربي. إذا أردت ملفاتك الخاصة، شغّل خلية الرفع التالية بدل هذه.

In [ ]:
import urllib.request, os
from PIL import Image
from IPython.display import Audio, display

PORTRAIT = 'portrait.jpg'
AUDIO = 'vo_test.wav'
try:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/Samir412005/video-lab/main/assets/portrait.jpg', PORTRAIT)
    urllib.request.urlretrieve('https://raw.githubusercontent.com/Samir412005/video-lab/main/assets/vo_test.wav', AUDIO)
    print('Downloaded demo files:', os.path.getsize(PORTRAIT), 'bytes image,',
          os.path.getsize(AUDIO), 'bytes audio')
    display(Image.open(PORTRAIT))
    display(Audio(AUDIO))
except Exception as e:
    print('Demo download failed:', e)
    print('Upload your own portrait + audio in the next cell instead.')

## 4b) (Optional) Upload your own portrait + audio
ارفع صورة بورتريه (وجه أمامي، إضاءة جيدة) وملف صوت WAV/MP3، ثم شغّل الخلية.

In [ ]:
from google.colab import files
uploaded = files.upload()
for name in uploaded:
    if name.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
        PORTRAIT = name
    elif name.lower().endswith(('.wav', '.mp3', '.m4a')):
        AUDIO = name
print('Portrait:', PORTRAIT)
print('Audio:', AUDIO)

## 5) Generate the talking head
`--still` = حركة رأس ثابتة نسبياً (أنسب للتعليق)، `--preprocess full` = قص الوجه تلقائياً، `--enhancer gfpgan` = تحسين الوجه، `--size 512` = جودة أعلى.

In [ ]:
!python inference.py --driven_audio {AUDIO} --source_image {PORTRAIT} --result_dir ./results --still --preprocess full --enhancer gfpgan --size 512

In [ ]:
# 6) Show the result
import glob
from IPython.display import Video, display
mp4s = sorted(glob.glob('./results/*.mp4'))
print('Results:', mp4s)
if mp4s:
    display(Video(mp4s[-1], width=512))

## Troubleshooting

- **No GPU / CUDA not available:** Runtime → Change runtime type → Hardware accelerator = **T4 GPU** → Save، ثم أعد التشغيل.
- **`ModuleNotFoundError: No module named 'kornia'`:** تثبيت المتطلبات توقّف عند `numpy==1.23.4` (لا توجد عجلة جاهزة له في بايثون 3.12/3.13) فلم تُثبَّت باقي الحزم. الحل: خلية التثبيت تستبعد أسطر `numpy` و`scipy` و`scikit-image` المثبّتة الإصدار من `requirements.txt` ثم تثبّت `numpy<2`.
- **`Failed to build 'basicsr'`:** حزمة basicsr 1.4.2 لا تُبنى في بيئة معزولة على بايثون 3.13 فيتوقّف التثبيت كلّه. الحل: التثبيت بـ `--no-build-isolation` (وتثبيت basicsr وحدها أولاً بنفس الخيار) فيبني ضد torch/setuptools المثبّتين أصلاً في Colab.
- **`AttributeError: module 'numpy' has no attribute 'VisibleDeprecationWarning'`:** إصدار NumPy الحديث لا يناسب SadTalker. الحل المجرّب: بيئة Conda ببايثون 3.8:
  ```
  !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  !bash /tmp/miniconda.sh -b -p /opt/conda
  !/opt/conda/bin/conda create -y -n sadtalker python=3.8
  !/opt/conda/bin/conda run -n sadtalker pip install -q torch==1.12.1+cu113 torchvision==0.13.1+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
  ```
  ثم نفّذ خلايا التثبيت والتوليد ببادئة `/opt/conda/bin/conda run -n sadtalker`.
- **GFPGAN بطيء أو فشل:** احذف `--enhancer gfpgan` من أمر التوليد.
- **الصوت طويل:** SadTalker يعمل بشكل أفضل على مقاطع 5–30 ثانية؛ قسّم التعليق الطويل إلى مقاطع.